File handling allows our python programs to read from and write to files stored on disk-essntial for persisting logs, loading configuration files, parsing CSV/JSON datasets, and handling file uploads in FastAPI

## The with open(...) Statement and File Modes

Always open files using the with statement (a context manager). It guarantees Python automatically closes the file and flushes data to disk the moment the intendented block finishes - even if an error occurs inside the block.

Always specify encoding="utf-8" so text behaves consistently across Linux, macOS, and windows




### Core File Modes:

- "r" (Read- default) : Opens an existing file for reading. Raises FileNotFoundError if missing.

- "w" (Write): Creates a new file or completely overwrites an existing file

- "a" (Append) : Opens a file and adds new content to the very end without erasing existing data.

- "x" (Exclusive Creation): Creates a new file, falling with FileExistsError if it already exists

In [ ]:
# Writing to a file (creates server.log or overwrites it)

with open ("server.log", mode="w", encoding="utf-8") as file:
    file.write("2027-11-18 10:00:00 | INFO | Server started\n")
    file.write("2027-11-18 10:01:12 | INFO | Connected to the PostgreSQL\n") 



In [2]:
# Appending a new line without losing previous logs

with open("server.log", mode="a", encoding="utf-8") as file:
    file.write("2026-11-18 10:05:44 | ERROR | Timeour on api/v1/orders\n")

# Reading Files: Full content vs Line-by-Line

Choose how you read based on file size:

- .read(): Loads the entire file into one str (best for small config files)


- Iterating directly (for line in file:): Reads one line at a time without loading the whole file into RAM-

critical when parsing multi-gigabyte server logs on Linux

In [3]:
# Method A: Read the entire file at once

with open("server.log", mode="r", encoding="utf-8") as file:
    full_content = file.read()
    print("___FUll log___")
    print(full_content)


# Method B: Stream line-by-line (memory-efficient)
print("___Filtered Errors___")
with open("server.log", mode="r", encoding="utf-8") as file:
    for line in file:
        # .strip() removes the trailing \n newline character
        clean_line = line.strip()
        if "ERROR" in clean_line:
            print(clean_line)

___FUll log___
2027-11-18 10:00:00 | INFO | Server started
2027-11-18 10:01:12 | INFO | Connected to the PostgreSQL
2026-11-18 10:05:44 | ERROR | Timeour on api/v1/orders

___Filtered Errors___
2026-11-18 10:05:44 | ERROR | Timeour on api/v1/orders


## Modern File Paths with pathlib.path

Instead of concatenating path strings with "/" manually, modern Python uses the built in pathlib.path object, which handles directory cration, existence checks, and quick reads/writes cleanly.

In [4]:
from pathlib import Path

#Define a folder and file path cleanly

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True) # Creates 'logs/' directory if it doesn't exist

app_log = log_dir/"app.txt" # Uses '// operator to join paths safely

# Quick write and read helpers for small files

app_log.write_text("Service healthy: 200 OK\n", encoding="utf-8")

if app_log.exists():
    print("Read via path:", app_log.read_text(encoding="utf-8").strip())

Read via path: Service healthy: 200 OK


# Real -world backend and API examples

### Example A : Reading and writing structured JSON (json module)

in backend development, configuration files and mock API databases are frequently stored as .json files.


Python's built-in json module converts directly between Python dictionaries/lists and JSON files.

In [ ]:
import json
from pathlib import Path

user_file = Path("users_store.json")

users_data = [
    {"id":1, "username": "devanath", "role": "intern"},
    {"id":2, "username": "alice", "role": "admin"},
]

# Save Python list/dict -> JSON folder_name: strile (indent=2 makes it human-radable)

with user_file.open(mode="w", encoding="utf-8") as f:
    json.dump(users_data, f,indent=2)

# Load JSON file -> Python list/dict;

with user_file.open(mode="r", encoding="utf-8") as f:
    loaded_users = json.load(f)


print("Loaded from JSON:", loaded_users[0]["username"])

Loaded from JSON: devanath


### Example B: Safe file loading with exception handling

Combining try/except with file operations prevents creashes when a configuration file is missing or corrupted.

In [11]:
import json
from pathlib import Path


def load_config(filepath: str) -> dict[str, str | int] :
    path = Path(filepath)

    try:
        with path.open(mode="r", encoding="utf-8") as f:
            return json.load(f)
    
    except FileNotFoundError:
        print(f"Warning : '{filepath} not found, Using default settings")
        return {"host": "127.0.0.1", "port": 8000}

    
    except json.JSONDecodeError:
        print(f"Error: {filepath} contains malformed JSON.")
        return {"host": "127.0.0.1", "port":8000}

config = load_config("missing_config.json")
print("Acitve config:", config)

Acitve config: {'host': '127.0.0.1', 'port': 8000}


In [3]:
print("""2026-11-19T09:00:01 | GET | /api/v1/health | 200 | 12.4
2026-11-19T09:01:15 | POST | /api/v1/login | 200 | 145.0
2026-11-19T09:02:30 | POST | /api/v1/login | 401 | 88.2
2026-11-19T09:03:10 | GET | /api/v1/users | 200 | 310.5
CORRUPTED_LINE_MISSING_FIELDS
2026-11-19T09:05:00 | GET | /api/v1/orders | 500 | 640.8
2026-11-19T09:06:22 | PUT | /api/v1/users | 200 | 190.0
2026-11-19T09:07:45 | GET | /api/v1/products | 200 | 95.3
2026-11-19T09:08:12 | DELETE | /api/v1/orders | 403 | 42.1
2026-11-19T09:09:00 | GET | /api/v1/orders | NOT_A_NUMBER | 120.0
2026-11-19T09:10:18 | GET | /api/v1/reports | 200 | 480.2
2026-11-19T09:11:05 | POST | /api/v1/orders | 201 | 215.6""")

2026-11-19T09:00:01 | GET | /api/v1/health | 200 | 12.4
2026-11-19T09:01:15 | POST | /api/v1/login | 200 | 145.0
2026-11-19T09:02:30 | POST | /api/v1/login | 401 | 88.2
2026-11-19T09:03:10 | GET | /api/v1/users | 200 | 310.5
CORRUPTED_LINE_MISSING_FIELDS
2026-11-19T09:05:00 | GET | /api/v1/orders | 500 | 640.8
2026-11-19T09:06:22 | PUT | /api/v1/users | 200 | 190.0
2026-11-19T09:07:45 | GET | /api/v1/products | 200 | 95.3
2026-11-19T09:08:12 | DELETE | /api/v1/orders | 403 | 42.1
2026-11-19T09:09:00 | GET | /api/v1/orders | NOT_A_NUMBER | 120.0
2026-11-19T09:10:18 | GET | /api/v1/reports | 200 | 480.2
2026-11-19T09:11:05 | POST | /api/v1/orders | 201 | 215.6


In [2]:
line="""2026-11-19T09:00:01 | GET | /api/v1/health | 200 | 12.4
2026-11-19T09:01:15 | POST | /api/v1/login | 200 | 145.0
2026-11-19T09:02:30 | POST | /api/v1/login | 401 | 88.2
2026-11-19T09:03:10 | GET | /api/v1/users | 200 | 310.5
CORRUPTED_LINE_MISSING_FIELDS
2026-11-19T09:05:00 | GET | /api/v1/orders | 500 | 640.8
2026-11-19T09:06:22 | PUT | /api/v1/users | 200 | 190.0
2026-11-19T09:07:45 | GET | /api/v1/products | 200 | 95.3
2026-11-19T09:08:12 | DELETE | /api/v1/orders | 403 | 42.1
2026-11-19T09:09:00 | GET | /api/v1/orders | NOT_A_NUMBER | 120.0
2026-11-19T09:10:18 | GET | /api/v1/reports | 200 | 480.2
2026-11-19T09:11:05 | POST | /api/v1/orders | 201 | 215.6"""


print(line)

2026-11-19T09:00:01 | GET | /api/v1/health | 200 | 12.4
2026-11-19T09:01:15 | POST | /api/v1/login | 200 | 145.0
2026-11-19T09:02:30 | POST | /api/v1/login | 401 | 88.2
2026-11-19T09:03:10 | GET | /api/v1/users | 200 | 310.5
CORRUPTED_LINE_MISSING_FIELDS
2026-11-19T09:05:00 | GET | /api/v1/orders | 500 | 640.8
2026-11-19T09:06:22 | PUT | /api/v1/users | 200 | 190.0
2026-11-19T09:07:45 | GET | /api/v1/products | 200 | 95.3
2026-11-19T09:08:12 | DELETE | /api/v1/orders | 403 | 42.1
2026-11-19T09:09:00 | GET | /api/v1/orders | NOT_A_NUMBER | 120.0
2026-11-19T09:10:18 | GET | /api/v1/reports | 200 | 480.2
2026-11-19T09:11:05 | POST | /api/v1/orders | 201 | 215.6
